In [1]:
import getpass
import os

os.environ["DEEPSEEK_API_KEY"] = getpass.getpass()

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model = "deepseek-flash")

 ········


In [2]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi! I'm Hiki")])

AIMessage(content='Hi Hiki! Nice to meet you 😊 What can I help you with today?', additional_kwargs={'refusal': None, 'reasoning_content': "We need answer. User just says Hi! I'm Hiki. Need respond friendly. Maybe ask how can help. No need tools. We can say Hi Hiki, nice to meet you. What can I do for you today? Keep concise."}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 36, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 51, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 36}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'e8de1f92-d074-41aa-ba5a-ada0518f97fa', 'finish_reason': 'stop', 'log

In [3]:
model.invoke([HumanMessage(content="What's my name?")])
# 我们可以看到它没有将之前的对话轮次作为上下文
# 因此无法回答该问题。 这会导致糟糕的聊天机器人体验!

AIMessage(content='I don’t know your name unless you tell me. What would you like me to call you?', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer. User asks "What\'s my name?" We don\'t know. Need respond maybe cannot know unless told. We should be honest. No personal data. Could say "I don\'t know your name—you haven\'t shared it with me. If you tell me, I can use it." Ensure not ask for sensitive? Fine.'}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 35, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 70, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'a

In [4]:
# 为了绕过这个问题，我们需要将整个对话历史传递给模型
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="Hi! I'm Hiki"),
        AIMessage(content="Hi Hiki! Nice to meet you. How can I help you today?"),
        HumanMessage(content="What's my name?"),
    ]
)
# 这是支撑聊天机器人进行对话交互的基本理念

AIMessage(content='Your name is Hiki! 😊', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer. User says "Hi! I\'m Hiki". Then asks "What\'s my name?" We know from conversation: Hiki. Need answer. Simple. Ensure maybe "Your name is Hiki." Could be playful. Final.'}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 61, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 50, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 61}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '6521d108-f3a9-43f2-9c81-7e5721db4116', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0c9ca-5794-7

In [5]:
# 消息历史
## 我们可以使用消息历史类来包装我们的模型，使其具有状态
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 它返回一个新的对象 with_message_history，
# 这个新对象和原来的 model 用法几乎一样（也能 .invoke() / .stream()），
# 但多了一层自动管理历史的逻辑
with_message_history = RunnableWithMessageHistory(model, get_session_history)

C:\Users\HikiLin\Desktop\LangChain-Learning\LangChain-Learning\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
config = {"configurable": {"session_id": "abc2"}}

In [7]:
# 把HumanMessage和session_id一起给with_message_history
response = with_message_history.invoke(
    [HumanMessage(content="Hi! I'm Hiki")],
    config=config,
)

response.content

'Hi Hiki! Nice to meet you 😊 What would you like to do or talk about today?'

In [8]:
# 再次发送同一个session_id
# with_message_history就会先看store中有没有session_id?
# 结果发现已经有了，于是它读取store中的历史并和新message一起发给model
# 然后把新的历史存入store
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is **Hiki** — you just told me. 😊'

In [9]:
# 如果更改配置，用不同的session_id，它就会开始新对话
config = {"configurable": {"session_id": "abc3"}}

response = with_message_history.invoke(
    [HumanMessage("What's my name?")],
    config=config,
)

response.content

'I don’t know your name—you haven’t told me yet. What should I call you?'

In [10]:
# Meanwhile,我们可以回到之前的对话
config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)
response.content

'Your name is **Hiki**.'

In [11]:
# 提示词模板
## 首先，让我们添加一个系统消息
## 为此，我们将创建一个 ChatPromptTemplate
## 我们将利用 MessagesPlaceholder 来传递所有消息
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [13]:
response = chain.invoke({"messages": [HumanMessage(content="hi! I'm bob")]})

response.content

'Hi Bob! Nice to meet you. How can I help you today?'

In [14]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

C:\Users\HikiLin\Desktop\LangChain-Learning\LangChain-Learning\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [15]:
config = {"configurable": {"session_id": "abc5"}}

In [19]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi! I'm Hiki")],
    config=config,
)

response.content

'Hi Hiki! Nice to meet you. 😊 What can I help you with today?'

In [20]:
response = with_message_history.invoke(
    [HumanMessage(content="What;s my name?")],
    config=config,
)

response.content

'Your name is Hiki. 😊'

In [22]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [23]:
response = chain.invoke(
    {"messages": [HumanMessage(content="hi! I'm bob")], "language": "chinese"}
)

response.content

'你好，Bob！很高兴认识你。有什么我可以帮你的吗？'

In [24]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

C:\Users\HikiLin\Desktop\LangChain-Learning\LangChain-Learning\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [25]:
config = {"configurable": {"session_id": "abc11"}}

In [26]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="hi! I'm todd")], "language": "chinese"},
    config=config,
)

response.content

'你好，Todd！很高兴认识你😊 有什么我可以帮你的吗？'

In [27]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Spanish"},
    config=config,
)

response.content

'Tu nombre es Todd. 😊'

In [29]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=65,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

NotImplementedError: get_num_tokens_from_messages() is not presently implemented for model deepseek-flash. See https://platform.openai.com/docs/guides/text-generation/managing-tokens for information on how messages are converted to tokens.